# Notebook 04 — Modelos Clásicos sobre Embeddings

**Objetivo:** Comparar 5 modelos clásicos de ML entrenados sobre los embeddings de 1024 dimensiones.

**Prerequisito:** `data/embeddings/{train,val,test}_X.npy` deben existir (notebook 03).

Modelos evaluados:
- Logistic Regression (paramétrico)
- K-Nearest Neighbors (no paramétrico)
- Random Forest (ensamble)
- MLP (red neuronal)
- SVM (máquina de soporte vectorial)

Todos usan **exactamente los mismos embeddings** — comparación justa.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

# COLAB — descomenta si ejecutas en Colab
# !git clone https://github.com/TU_USUARIO/Malaria-Dectetion-Deeplearning.git
# %cd Malaria-Dectetion-Deeplearning
# !pip install -r requirements.txt -q

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_embeddings, save_json
from src.training.train_classical import train_single_model
from src.evaluation.confusion import plot_confusion_matrix
from src.visualization.training_plots import plot_metrics_comparison

set_global_seed(42)
%matplotlib inline

In [ ]:
cfg = load_config('configs/classical.yaml')
X_train, y_train = load_embeddings('train', cfg['embeddings_dir'])
X_val,   y_val   = load_embeddings('val',   cfg['embeddings_dir'])
X_test,  y_test  = load_embeddings('test',  cfg['embeddings_dir'])
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
from pathlib import Path
out_dir = Path(cfg['output_dir'])
fig_dir = Path(cfg['figures_dir'])
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

all_results = {}
trained_models = {}

for model_name, model_cfg in cfg['models'].items():
    if not model_cfg.get('enabled', True):
        continue
    print(f'\n▶ Entrenando: {model_name}...')
    results, model = train_single_model(
        model_name=model_name, model_cfg=model_cfg,
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test,
        cv_folds=cfg['cv_folds'], seed=cfg['seed'],
        bootstrap_cfg=cfg['bootstrap'],
    )
    all_results[model_name] = results
    trained_models[model_name] = model
    
    test_acc = results['splits']['test']['accuracy']
    test_f1  = results['splits']['test']['f1_macro']
    test_auc = results['splits']['test'].get('roc_auc', float('nan'))
    print(f'  acc={test_acc:.4f} | f1={test_f1:.4f} | AUC={test_auc:.4f}')
    print(f'  Mejor params: {results["best_params"]}')

In [ ]:
# Guardar resultados y modelos
for name, r in all_results.items():
    save_json(r, out_dir / f'{name}.json')

Path('artifacts/checkpoints').mkdir(parents=True, exist_ok=True)
with open('artifacts/checkpoints/classical_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)
print('Modelos y métricas guardados.')

In [ ]:
# Tabla comparativa
rows = []
for name, r in all_results.items():
    test = r['splits']['test']
    ci   = test.get('bootstrap_ci_95', {})
    rows.append({
        'Modelo': name,
        'Train Acc': round(r['splits']['train']['accuracy'], 4),
        'Val Acc':   round(r['splits']['val']['accuracy'],   4),
        'Test Acc':  round(test['accuracy'], 4),
        'Test F1':   round(test['f1_macro'], 4),
        'Test AUC':  round(test.get('roc_auc', float('nan')), 4),
        'Test BAcc': round(test['balanced_accuracy'], 4),
        'CI lower':  round(ci.get('accuracy', {}).get('lower', float('nan')), 4),
        'CI upper':  round(ci.get('accuracy', {}).get('upper', float('nan')), 4),
    })

df_results = pd.DataFrame(rows).sort_values('Test F1', ascending=False)
print('\n=== TABLA COMPARATIVA ===')
display(df_results)
df_results.to_csv(out_dir / 'comparison_table.csv', index=False)

In [ ]:
# Barplot comparativo
summary = {name: {'test_f1_macro': all_results[name]['splits']['test']['f1_macro']} for name in all_results}
fig = plot_metrics_comparison(summary, metric='test_f1_macro',
                               save_path='artifacts/figures/models_comparison.png')
plt.show()

In [ ]:
# Matrices de confusión
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    fig = plot_confusion_matrix(y_test, y_pred, model_name=name,
                                 save_path=f'artifacts/figures/cm_{name}.png')
    plt.show()
    plt.close()

In [ ]:
# Curvas ROC
from sklearn.metrics import roc_curve, auc
fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.cm.tab10.colors

for i, (name, model) in enumerate(trained_models.items()):
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i % 10], lw=2, label=f'{name} (AUC={roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC — Test')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()